# Deployment

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### Model Training

1.  #### Data prepartion

In [7]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

  - #### Data Loading

In [8]:
df = pd.read_csv(data)

  - #### Data Formating

In [9]:
def format(element):
  return element.str.lower().str.replace(" ", "_")


In [10]:
df.columns = format(df.columns)
df.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

In [11]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
# categorical_columns

In [12]:
df[categorical_columns] = df[categorical_columns].apply(format)

In [13]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [14]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors="coerce")
df.totalcharges = df.totalcharges.fillna(0)

# df.dtypes

In [15]:
df.churn = (df.churn == "yes").astype(int)

In [16]:
dv = DictVectorizer()

  - #### Data Splitting

In [17]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

2. #### Model Training

In [18]:
def train(df, y_train, C=1.0):
  dicts = df[numerical + categorical].to_dict(orient="records")

  dv = DictVectorizer(sparse=False)
  X_train = dv.fit_transform(dicts)

  model = LogisticRegression(C=C, max_iter=1000)
  model.fit(X_train, y_train)

  return dv, model

3. #### Model Prediction

In [19]:
def predict(df, dv, model):
  dicts = df[numerical + categorical].to_dict(orient="records")

  X = dv.transform(dicts)

  y_pred = model.predict_proba(X)[: ,1]

  return y_pred

4. #### Model Evluation

In [20]:
C = 1.0
n_splits = 5

In [21]:
# TODO

kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

C=1.0 0.842 +- 0.007


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
scores

[np.float64(0.8446829053857807),
 np.float64(0.8453826479936432),
 np.float64(0.8333774834437085),
 np.float64(0.8347968759992326),
 np.float64(0.8517461014715573)]

In [23]:
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

np.float64(0.8584032088573997)

### Model Saving

In [24]:
import pickle

In [25]:
output_file = "modelC=%s.bin" % C
output_file

'modelC=1.0.bin'

In [26]:
# f_out = open(output_file, "wb")
# pickle.dump((dv, model), output_file)
# f_out.close()

In [ ]:
# Closes the file automatically
# with open(output_file, "wb") as f_out:
#   pickle.dump((dv, model), f_out)

### Model Loading

In [1]:
import pickle

In [2]:
model_file = "modelC=1.0.bin"

In [3]:
with open(model_file, "rb") as f_in:
  (dv, model) = pickle.load(f_in)

In [4]:
dv, model

(DictVectorizer(sparse=False), LogisticRegression(max_iter=1000))

### Making Request

In [16]:
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [17]:
X_customer = dv.transform([customer])
# X_customer

In [18]:
model.predict_proba(X_customer)[0, 1]

np.float64(0.6275953527536646)

## Web Service With Flask
- **Web Service**: A web service is a way for applications to talk to each other over HTTP - the same protocol a browser uses to load websites. One application (a client) sends a request to an address (a URL), and the other application (the server) sends back a response.

## Virtual Environment with Pipenv
- **Virtual Environment**: creating a virtual environment for a projrct or service with the required  dependencies and their version to assure reproducability and avoid conflicts with tje host installed dependencies. 
- **Pipenv**: 
```bash
pipenv install pandas scikit-learn==2.0.24

pipenv install

pipenv shell
```

## Containerazation with Docker
- **Docker**: allows to have more isolation by creating a container for each service instead of a virtual environment